In [72]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
import tqdm
import sys
import os
import pandas as pd
import time

In [74]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("clip-ViT-B-32")

2026-04-24 15:40:52,213 - INFO - Use pytorch device_name: cpu
2026-04-24 15:40:52,214 - INFO - Load pretrained SentenceTransformer: clip-ViT-B-32
2026-04-24 15:40:52,389 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/clip-ViT-B-32/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-24 15:40:52,402 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/clip-ViT-B-32/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/modules.json "HTTP/1.1 200 OK"
2026-04-24 15:40:52,531 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/clip-ViT-B-32/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-24 15:40:52,550 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/clip-ViT-B-32/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-24 15:40:52,678 - INFO - HTTP Request: HEAD ht

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /home/kantz/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-24 15:40:53,599 - INFO - HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/clip-ViT-B-32 "HTTP/1.1 200 OK"


In [75]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [76]:
sys.path.append("..")

In [77]:
from utils.datasets import DBPedia
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import numpy as np
from utils.datasets.base_dataset import DataTensor

In [78]:
dataset = DBPedia(base_dir=Path("../data/dbpedia"))

In [79]:
test_label = "horse on the bow"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [80]:
test_tensor.to_literal().n3()

'"{\\"data\\": [-0.04262268915772438, -0.1333884298801422, 0.08612467348575592, -0.02091798186302185, -0.15041583776474, 0.1737712323665619, -0.348517507314682, -0.9152146577835083, -0.3341490924358368, 0.17787908017635345, 0.05680551752448082, -0.20725640654563904, 0.5095284581184387, 0.18332251906394958, 0.11725624650716782, 0.1168459951877594, 0.20500661432743073, -0.1478302776813507, -0.007135884836316109, 0.4961417019367218, 0.3664361536502838, 0.41410648822784424, -0.04107498377561569, 0.5009220838546753, -0.2138804793357849, -0.23931393027305603, -0.7052134275436401, 0.4793880879878998, -0.17963360249996185, 0.2792432904243469, -0.06826944649219513, 0.0803503543138504, -0.20019735395908356, 0.25140535831451416, -0.16235408186912537, 0.1176009476184845, 0.10282524675130844, 0.20387926697731018, -0.04952261596918106, 0.26974374055862427, 0.13957971334457397, 0.5327335596084595, 0.12486020475625992, -0.08572806417942047, 0.1065823957324028, -0.06904290616512299, 0.19244082272052765

In [81]:
db_qlever = QleverDBNative(
    id="qlever",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    name="QLever",
    port_offset=-100
)
db_no_tensor_idx = QleverDBNative(
    id="qlever",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    enable_tensor_index=False,
    name="QLever (No Tensor Vocabulary)",
    port_offset=-200
)
db_fuseki = FusekiDBNative(
    id="fuseki",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    name="Fuseki",
    exec_dir="../../jena-datatensor",
    port_offset=-300
)
possible_queries = db_qlever.get_queries(test_tensor)
print(possible_queries)
dbs: list[QleverDBNative | FusekiDBNative] = [db_qlever, db_no_tensor_idx, db_fuseki]
indices = ["index-encoded", "index-encoded-no-tidx", "fuseki-encoded"]
ids = ["dbpedia-encoded-tidx", "dbpedia-encoded", None]
for db, index, id in zip(dbs, indices, ids):
    db.db_dir = Path("../data/dbpedia") / index
    db.id = id if id is not None else db.id

2026-04-24 15:40:55,060 - WARNING - Killing any existing process using port 25946 before starting the server
2026-04-24 15:40:55,079 - ERROR - Command failed with return code 1
2026-04-24 15:40:55,080 - INFO - Initialized QLeverDBNative with id=qlever, port_id=25946, dataset=DBPedia, name=QLever, use_encoded_ttl=True, endpoint=http://localhost:25946/qlever-with-tidx/sparql
2026-04-24 15:40:55,081 - WARNING - Killing any existing process using port 25847 before starting the server
2026-04-24 15:40:55,096 - ERROR - Command failed with return code 1
2026-04-24 15:40:55,096 - INFO - Initialized QLeverDBNative with id=qlever, port_id=25847, dataset=DBPedia, name=QLever (No Tensor Vocabulary), use_encoded_ttl=True, endpoint=http://localhost:25847/qlever-no-tidx/sparql
2026-04-24 15:40:55,097 - WARNING - Killing any existing process using port 28748 before starting the server
2026-04-24 15:40:55,112 - ERROR - Command failed with return code 1


{<QUERY_DIFFICULTY.EASY: 'easy'>: {<QUERY_TYPE.EMBEDDED: 'embedded'>: '\nPREFIX dbr: <http://dbpedia.org/resource/>\nPREFIX dbo: <http://dbpedia.org/ontology/>\nPREFIX dtf: <https://w3id.org/rdf-tensor/functions#>\nSELECT DISTINCT ?s ?thumb_emb ?dist WHERE {\n    ?s a dbo:Ship ;\n         dbo:thumbnail_embedding ?thumb_emb .\n    BIND(dtf:cosineSimilarity(?thumb_emb, "{\\"data\\": [-0.04262268915772438, -0.1333884298801422, 0.08612467348575592, -0.02091798186302185, -0.15041583776474, 0.1737712323665619, -0.348517507314682, -0.9152146577835083, -0.3341490924358368, 0.17787908017635345, 0.05680551752448082, -0.20725640654563904, 0.5095284581184387, 0.18332251906394958, 0.11725624650716782, 0.1168459951877594, 0.20500661432743073, -0.1478302776813507, -0.007135884836316109, 0.4961417019367218, 0.3664361536502838, 0.41410648822784424, -0.04107498377561569, 0.5009220838546753, -0.2138804793357849, -0.23931393027305603, -0.7052134275436401, 0.4793880879878998, -0.17963360249996185, 0.279243

In [82]:
triple_counts = []
for db in dbs[:-1]:
    with db:
        count = db.get_triple_count()
        count_tensors = db.query("""
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(?v) AS ?count) WHERE {
    ?s dbo:thumbnail_embedding ?v .
}""")["count"].values[0]
        print(f"{db.name}: {count} triples, {count_tensors} tensors")
        triple_counts.append({
            "count": count,
            "tensors": count_tensors,
            "db": db.name
        })
print(f"Total triples in DB: {sum([tc['count'] for tc in triple_counts])}")

2026-04-24 15:40:55,191 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 15:40:55,191 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 15:40:55,192 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 15:40:55,192 - INFO - Stopping server!


2026-04-24 15:40:55,207 - ERROR - Command failed with return code 1
2026-04-24 15:40:55,207 - INFO - Starting QLever server on port 25946
2026-04-24 15:40:55,208 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 15:40:55,210 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-04-24 15:40:56,211 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-04-24 15:40:57,212 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-04-24 15:40:58,214 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-04-24 15:40:59,223 - I

QLever: 421770566 triples, 2329464 tensors


2026-04-24 15:41:01,422 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25847/qlever-no-tidx/sparql)
2026-04-24 15:41:02,423 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25847/qlever-no-tidx/sparql)
2026-04-24 15:41:03,425 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25847/qlever-no-tidx/sparql)
2026-04-24 15:41:04,436 - INFO - Server is up and responding to queries
2026-04-24 15:41:05,528 - INFO - Stopping server!
2026-04-24 15:41:05,608 - ERROR - Command failed with return code 1


QLever (No Tensor Vocabulary): 421770566 triples, 2329464 tensors
Total triples in DB: 843541132


In [83]:
counts_df = pd.DataFrame(triple_counts)
counts_df

,count,tensors,db
0,421770566,2329464,QLever
1,421770566,2329464,QLever (No Tensor Vocabulary)


In [84]:
counts_df["power"] = counts_df["count"].apply(lambda x: int(np.log10(x)))
counts_df["size"] = counts_df["count"]
counts_df["full_size"] = counts_df["count"]
counts_df["full_number_of_tensors"] = counts_df["tensors"]

In [85]:
from utils.helpers import pretty_print_counts

out_path_counts = Path("../scratch/results") / "dbpedia_counts.tex"
pretty_print_counts(counts_df, out_path_counts)

['power', 'size', 'full_size', 'full_number_of_tensors'] []


,Power,Generation $t$,$n$,$n_{tensors}$
0,8,421770566,421770566,2329464
1,8,421770566,421770566,2329464


In [104]:
with db_qlever as db:
    timings = []
    for _ in tqdm.tqdm(range(5)):
        start = time.time()
        r = db.query_auto(
            query_difficulty=QUERY_DIFFICULTY.EASY,
            query_type=QUERY_TYPE.INDEX,
            tensor=test_tensor,
        )
        end = time.time()
        timings.append(end - start)
print("Timings avg:", sum(timings) / len(timings))
r

2026-04-24 16:10:41,170 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 16:10:41,170 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 16:10:41,171 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 16:10:41,171 - INFO - Stopping server!
2026-04-24 16:10:41,189 - ERROR - Command failed with return code 1
2026-04-24 16:10:41,189 - INFO - Starting QLever server on port 25946
2026-04-24 16:10:41,190 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 16:10:41,191 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-04-24 16:10:42,192 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/spar

Timings avg: 0.11167473793029785


,s,thumb_emb,dist
0,dbr:SS_Albert_M._Boe,"{""data"": [0.3019237518310547, -0.1744167208671...",27.03730392456
1,dbr:MV_Uhuru,"{""data"": [0.10303764045238495, -0.229101330041...",27.02251815796
2,dbr:Original_six_frigates_of_the_United_States...,"{""data"": [0.3499048352241516, -0.0667797625064...",26.38556671143
3,dbr:Fire-float_Pyronaut,"{""data"": [0.5736645460128784, 0.02624748647212...",26.04682922363
4,dbr:Eppleton_Hall_(1914),"{""data"": [0.15651816129684448, -0.074189648032...",25.96257019043
5,dbr:Sherman_Zwicker,"{""data"": [0.050441011786460876, -0.30955773591...",25.90690803528
6,dbr:PS_Comet,"{""data"": [0.3139987587928772, 0.04801601171493...",25.84019470215
7,dbr:Ross_Tiger,"{""data"": [0.40394482016563416, -0.303305000066...",25.53351593018
8,dbr:NLV_Pole_Star,"{""data"": [0.4585147500038147, 0.00598367303609...",25.43468475342
9,dbr:Kathleen_and_May,"{""data"": [0.24212899804115295, 0.1124063432216...",25.43309020996


In [16]:
with dbs[0] as db:
    res_native = db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
SELECT * WHERE {{


    ?s a dbo:Ship .
    ?s dbo:thumbnail_embedding ?thumb_emb .
    ?s dbo:thumbnail_original ?thumb .      

    BIND(dtf:dotProduct(?thumb_emb, {test_tensor.to_literal().n3()}) AS ?dist)
}} 

ORDER BY DESC(?dist)
LIMIT 10""")
res_native

2026-04-24 14:40:08,474 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 14:40:08,475 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 14:40:08,475 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 14:40:08,475 - INFO - Stopping server!
2026-04-24 14:40:08,554 - ERROR - Command failed with return code 1
2026-04-24 14:40:08,555 - INFO - Starting QLever server on port 25943
2026-04-24 14:40:08,555 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:40:08,557 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:40:09,558 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

,s,thumb_emb,thumb,dist
0,dbr:HMS_Trent_(1757),"{""data"": [0.20562176406383514, 0.2037418782711...",https://upload.wikimedia.org/wikipedia/commons...,31.31344604492
1,dbr:Fram,"{""data"": [0.25629281997680664, -0.140227705240...",https://upload.wikimedia.org/wikipedia/commons...,29.51948547363
2,dbr:Fram,"{""data"": [0.25629281997680664, -0.140227705240...",https://upload.wikimedia.org/wikipedia/commons...,29.51948547363
3,dbr:Fram,"{""data"": [0.25629281997680664, -0.140227705240...",https://upload.wikimedia.org/wikipedia/commons...,29.51948547363
4,dbr:Fram,"{""data"": [0.25629281997680664, -0.140227705240...",https://upload.wikimedia.org/wikipedia/commons...,29.51948547363
5,dbr:Fram,"{""data"": [0.25629281997680664, -0.140227705240...",http://upload.wikimedia.org/wikipedia/commons/...,29.51948547363
6,dbr:Fram,"{""data"": [0.25629281997680664, -0.140227705240...",http://upload.wikimedia.org/wikipedia/commons/...,29.51948547363
7,dbr:Spanish_ship_San_Ildefonso,"{""data"": [-0.08917225152254105, -0.20555926859...",https://upload.wikimedia.org/wikipedia/commons...,29.03779411316
8,dbr:Spanish_ship_San_Ildefonso,"{""data"": [-0.08917225152254105, -0.20555926859...",https://upload.wikimedia.org/wikipedia/commons...,29.03779411316
9,dbr:Margaret_Todd_(schooner),"{""data"": [0.506389856338501, 0.253706932067871...",https://upload.wikimedia.org/wikipedia/commons...,28.92206573486


In [ ]:
with dbs[0] as db:
    res =db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT * WHERE {{
SERVICE tensorIndex: {{
    _:config tensorIndex:numNN 10 ;
    tensorIndex:left ?query_vector ;
    tensorIndex:bindDistance ?dist ;
    tensorIndex:payload ?s, ?thumb ;
    tensorIndex:searchK 64 ;
    tensorIndex:kIVF 128 ;
    # tensorIndex:experimentalRightCacheName "easy_index_dbpedia" ;
    tensorIndex:right ?thumb_emb ;
    tensorIndex:algorithm tensorIndex:ivf ;
    tensorIndex:distance tensorIndex:dot .
       {{
            ?s a dbo:Ship ;
            dbo:thumbnail_embedding ?thumb_emb ;
            dbo:thumbnail_original ?thumb .
        }}
    }}
    VALUES (?query_vector) {{ ({test_tensor.to_literal().n3()}) }}
}} 
ORDER BY DESC(?dist)
""")
res

2026-04-24 14:40:48,643 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 14:40:48,643 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 14:40:48,644 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 14:40:48,644 - INFO - Stopping server!
2026-04-24 14:40:48,732 - ERROR - Command failed with return code 1
2026-04-24 14:40:48,732 - INFO - Starting QLever server on port 25943
2026-04-24 14:40:48,732 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:40:48,734 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:40:49,735 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

,query_vector,dist,s,thumb,thumb_emb
0,"{""data"": [-0.04262268915772438, -0.13338842988...",29.51948547363,dbr:Fram,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.25629281997680664, -0.140227705240..."
1,"{""data"": [-0.04262268915772438, -0.13338842988...",29.51948547363,dbr:Fram,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.25629281997680664, -0.140227705240..."
2,"{""data"": [-0.04262268915772438, -0.13338842988...",29.51948547363,dbr:Fram,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.25629281997680664, -0.140227705240..."
3,"{""data"": [-0.04262268915772438, -0.13338842988...",29.51948547363,dbr:Fram,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.25629281997680664, -0.140227705240..."
4,"{""data"": [-0.04262268915772438, -0.13338842988...",29.51948547363,dbr:Fram,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [0.25629281997680664, -0.140227705240..."
5,"{""data"": [-0.04262268915772438, -0.13338842988...",29.51948547363,dbr:Fram,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [0.25629281997680664, -0.140227705240..."
6,"{""data"": [-0.04262268915772438, -0.13338842988...",28.92206382751,dbr:Margaret_Todd_(schooner),https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.506389856338501, 0.253706932067871..."
7,"{""data"": [-0.04262268915772438, -0.13338842988...",28.92206382751,dbr:Margaret_Todd_(schooner),https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.506389856338501, 0.253706932067871..."
8,"{""data"": [-0.04262268915772438, -0.13338842988...",28.79962348938,dbr:La_Grace,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.3559197187423706, 0.05999966338276..."
9,"{""data"": [-0.04262268915772438, -0.13338842988...",28.79962348938,dbr:La_Grace,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [0.3559197187423706, 0.05999966338276..."


In [20]:
res.to_csv(Path("scratch") / "dbpedia_results_index.csv", index=False)

In [21]:
with dbs[1] as db:
    db.query_auto(test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.HARD)

2026-04-24 14:40:56,056 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-no-tidx/qlever-no-tidx_run.log
2026-04-24 14:40:56,057 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 14:40:56,057 - WARNING - DB directory ../data/dbpedia/index-encoded-no-tidx already exists!
2026-04-24 14:40:56,058 - INFO - Stopping server!
2026-04-24 14:40:56,142 - ERROR - Command failed with return code 1
2026-04-24 14:40:56,143 - INFO - Starting QLever server on port 25844
2026-04-24 14:40:56,143 - INFO - Running command: qlever-server -i dbpedia-encoded --port 25844 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:40:56,146 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2026-04-24 14:40:57,147 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2

In [24]:
from utils.helpers import ndcgscore_query
reference_result = res_native
score = ndcgscore_query(res, reference_result, k=10)
print(f"Score: {score}")

Score: 0.8042472959280005


In [25]:
# count ships
with dbs[0] as db:
    rs =db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(*) AS ?count) WHERE {{ 
            ?s a dbo:Ship ;
            dbo:thumbnail_embedding ?thumb_emb ;
            dbo:thumbnail_original ?thumb  .
}} LIMIT 10""")
    rr =  db.query(f"""PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(*) AS ?count) WHERE {{ 
            ?s a dbo:RailwayLine ;
            dbo:thumbnail_embedding ?thumb_emb ;
            dbo:thumbnail_original ?thumb  .
}} LIMIT 10""")
rs.iloc[0], rr.iloc[0]

2026-04-24 14:41:25,772 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 14:41:25,772 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 14:41:25,773 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 14:41:25,773 - INFO - Stopping server!
2026-04-24 14:41:25,854 - ERROR - Command failed with return code 1
2026-04-24 14:41:25,855 - INFO - Starting QLever server on port 25943
2026-04-24 14:41:25,855 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:41:25,857 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:41:26,858 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

(count    70312
 Name: 0, dtype: str,
 count    39797
 Name: 0, dtype: str)

In [ ]:
# for find similar thumbnails between ships and railroads
times = []
with dbs[0] as db:
    #     warmup = db.query("""
    # PREFIX dbr: <http://dbpedia.org/resource/>
    # PREFIX dbo: <http://dbpedia.org/ontology/>
    # PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
    # SELECT * WHERE {
    #     { SELECT * WHERE {
    #         ?r a dbo:RailwayLine .
    #         ?r dbo:thumbnail_embedding ?thumb_rail_emb .
    #         ?r dbo:thumbnail_original ?thumb_rail .
    #     } LIMIT 1
    #     }
    #     SERVICE tensorIndex: {
    #     _:config tensorIndex:numNN 10 ;
    #     tensorIndex:left ?thumb_rail_emb ;
    #     tensorIndex:bindDistance ?dist ;
    #     tensorIndex:payload ?s, ?thumb_ship ;
    #     tensorIndex:searchK 1 ;
    #     tensorIndex:nTrees 128 ;
    #     tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
    #     tensorIndex:right ?thumb_emb_ship ;
    #     tensorIndex:algorithm tensorIndex:ivf ;
    #     tensorIndex:distance tensorIndex:dot .
    #         {
    #             ?s a dbo:Ship ;
    #             dbo:thumbnail_embedding ?thumb_emb_ship ;
    #             dbo:thumbnail_original ?thumb_ship .
    #         }
    #     }
    #     }""")
    for _ in range(0):
        start = time.time()

        noised_tensor = DataTensor.from_numpy(
            test_tensor.data + np.random.normal(scale=0.001, size=test_tensor.shape)
        )
        railway_ship_assoc = db.query(f"""
    PREFIX dbr: <http://dbpedia.org/resource/>
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
    PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
    SELECT DISTINCT ?r ?s  ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
        {{
        SELECT DISTINCT ?r ?s ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
            
            ?r a dbo:RailwayLine .
            ?r dbo:thumbnail_embedding ?thumb_rail_emb .
                                        
            SERVICE tensorIndex: {{
            _:config tensorIndex:numNN 1 ;
            tensorIndex:left ?thumb_rail_emb ;
            tensorIndex:bindDistance ?dist ;
            tensorIndex:payload ?s, ?thumb_ship ;
            tensorIndex:searchK 1 ;
            tensorIndex:nTrees 512 ;
            tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
            tensorIndex:right ?thumb_ship_emb ;
            tensorIndex:algorithm tensorIndex:ivf ;
            tensorIndex:distance tensorIndex:dot .
            {{
                ?s a dbo:Ship ;
                dbo:thumbnail_embedding ?thumb_ship_emb ;    
            }}
            }}                  
        }}
        }}
        VALUES (?some_emb) {{ ({noised_tensor.to_literal().n3()}) }}
                                    
    }}
    ORDER BY DESC(?dist)
    LIMIT 20""")
        end = time.time()
        delta = end - start
        times.append(delta)
times

2026-04-24 14:42:04,041 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 14:42:04,041 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 14:42:04,042 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 14:42:04,042 - INFO - Stopping server!
2026-04-24 14:42:04,124 - ERROR - Command failed with return code 1
2026-04-24 14:42:04,125 - INFO - Starting QLever server on port 25943
2026-04-24 14:42:04,125 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:42:04,127 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:42:05,128 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

[]

In [ ]:
with db_qlever as db:
    hard_results = db.query(
        f"""
        PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT DISTINCT ?r ?s  ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
    {{
    SELECT DISTINCT ?r ?s ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
        
        ?r a dbo:RailwayLine ;
         dbo:thumbnail_embedding ?thumb_rail_emb ;
         dbo:thumbnail_original ?thumb_rail_original .
        BIND(LCASE(STR(?thumb_rail_original)) AS ?thumb_rail)
        FILTER(STRENDS(?thumb_rail, ".png") || STRENDS(?thumb_rail, ".jpg") || STRENDS(?thumb_rail, ".jpeg")) .
                                    
        SERVICE tensorIndex: {{
        _:config tensorIndex:numNN 1 ;
        tensorIndex:left ?thumb_rail_emb ;
        tensorIndex:bindDistance ?dist ;
        tensorIndex:payload ?s, ?thumb_ship_emb ;
        tensorIndex:searchK 1 ;
        tensorIndex:kIVF 512 ;
        tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
        tensorIndex:right ?thumb_ship_emb ;
        tensorIndex:algorithm tensorIndex:ivf ;
        tensorIndex:distance tensorIndex:dot .
        {{
            ?s a dbo:Ship ;
            dbo:thumbnail_embedding ?thumb_ship_emb ;  
            dbo:thumbnail_original ?thumb_ship_original .
            BIND(LCASE(STR(?thumb_ship_original)) AS ?thumb_ship)
            FILTER(STRENDS(?thumb_ship, ".png") || STRENDS(?thumb_ship, ".jpg") || STRENDS(?thumb_ship, ".jpeg")) .
        }}
        }}                  
    }}
    }}
    VALUES (?some_emb) {{ ({test_tensor.to_literal().n3()}) }}         
}}
ORDER BY DESC(?dist)
LIMIT 10"""
    )
    easy_results = db.query_auto(
        test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.EASY
    )

2026-04-24 14:42:12,628 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 14:42:12,628 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-24 14:42:12,629 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 14:42:12,629 - INFO - Stopping server!
2026-04-24 14:42:12,711 - ERROR - Command failed with return code 1
2026-04-24 14:42:12,712 - INFO - Starting QLever server on port 25943
2026-04-24 14:42:12,712 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:42:12,714 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:42:13,715 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

In [30]:
easy_results.to_csv(Path("scratch") / "dbpedia_easy_results.csv", index=False)
easy_results

,s,thumb_emb,dist
0,dbr:Plastiki,"{""data"": [0.2111155092716217, -0.3721050918102...",27.98868370056
1,dbr:La_Recouvrance_(schooner),"{""data"": [0.17521078884601593, -0.134947225451...",27.91348266602
2,dbr:Grace_Quan,"{""data"": [0.4119803309440613, -0.0349842160940...",27.79566192627
3,dbr:Naga_Pelangi,"{""data"": [0.3078002333641052, 0.20805418491363...",27.02926635742
4,dbr:MV_Uhuru,"{""data"": [0.10303764045238495, -0.229101330041...",27.02251815796
5,dbr:Original_six_frigates_of_the_United_States...,"{""data"": [0.3499048352241516, -0.0667797625064...",26.38556671143
6,dbr:T._J._Potter,"{""data"": [0.18704546988010406, 0.1744214743375...",26.16094970703
7,dbr:Rainbow_Warrior_(1957),"{""data"": [0.28070318698883057, -0.729847669601...",25.84356880188
8,dbr:PS_Comet,"{""data"": [0.3139987587928772, 0.04801601171493...",25.84019470215
9,dbr:Gorch_Fock_(1933),"{""data"": [0.08817720413208008, -0.622045218944...",25.55829238892


In [31]:
easy_results.loc[0, "s"], easy_results.loc[0, "thumb_emb"], easy_results.loc[0, "dist"]

('dbr:Plastiki',
 '{"data": [0.2111155092716217, -0.37210509181022644, -0.1256008744239807, 0.3913021981716156, 0.07871392369270325, -0.07938449829816818, 0.11866770684719086, 0.23398229479789734, 0.39779090881347656, -0.16861432790756226, -0.004371523857116699, -0.048737335950136185, 0.4543386399745941, 0.21892008185386658, -0.34641119837760925, -0.2010156363248825, 0.2572568655014038, 0.06256873905658722, 0.26309633255004883, 0.5121279954910278, -0.2595289349555969, 0.06514829397201538, -0.2721770107746124, 0.18661709129810333, 0.7871850728988647, 0.3414989709854126, -0.27971822023391724, 0.42073994874954224, 0.007892824709415436, 0.2540137767791748, 0.03973083198070526, 0.13429725170135498, -0.17217501997947693, 0.4178314805030823, -0.7641716599464417, -0.22243069112300873, -0.2509402632713318, 0.09269417822360992, 0.07116783410310745, 0.7140947580337524, 0.0005893222987651825, -0.07359926402568817, 0.004615098237991333, 0.39646783471107483, -0.10759720206260681, -1.586665391921997,

In [32]:
hard_results

,r,s,dist,thumb_rail_emb,thumb_ship_emb
0,dbr:Chengdu_Metro,dbr:Virginia-class_submarine,149.7555847168,"{""data"": [0.22622749209403992, 0.0546034500002...","{""data"": [-0.10002894699573517, -0.08042059838..."
1,dbr:Taipa_line,dbr:Virginia-class_submarine,148.5448455811,"{""data"": [0.03102342039346695, -0.059559382498...","{""data"": [-0.10002894699573517, -0.08042059838..."
2,dbr:Yangluo_Line,dbr:Virginia-class_submarine,146.525604248,"{""data"": [0.13901150226593018, 0.1055586785078...","{""data"": [-0.10002894699573517, -0.08042059838..."
3,dbr:Buenos_Aires_Underground,dbr:Virginia-class_submarine,140.4030609131,"{""data"": [-0.05835258960723877, -0.16195827722...","{""data"": [-0.10002894699573517, -0.08042059838..."
4,dbr:Line_13_(CPTM),dbr:Virginia-class_submarine,139.5791625977,"{""data"": [-0.21847796440124512, -0.04886087775...","{""data"": [-0.10002894699573517, -0.08042059838..."
5,dbr:Kolkata_Metro,dbr:I-201-class_submarine,139.0804901123,"{""data"": [-0.0022863298654556274, -0.211197406...","{""data"": [0.0012263432145118713, -0.0422745048..."
6,dbr:Orlyval,dbr:Virginia-class_submarine,138.4921875,"{""data"": [0.11910116672515869, -0.394591093063...","{""data"": [-0.10002894699573517, -0.08042059838..."
7,dbr:Abbey_Line,dbr:Virginia-class_submarine,138.0496826172,"{""data"": [0.18120601773262024, -0.300545185804...","{""data"": [-0.10002894699573517, -0.08042059838..."
8,dbr:London_Overground,dbr:Virginia-class_submarine,137.636428833,"{""data"": [0.02959146350622177, 0.0380432158708...","{""data"": [-0.10002894699573517, -0.08042059838..."
9,dbr:CDGVAL,dbr:Virginia-class_submarine,135.4937896729,"{""data"": [-0.1729656457901001, 0.0039286464452...","{""data"": [-0.10002894699573517, -0.08042059838..."


In [33]:
# load images for both results
from utils.dbs.base_db import BaseDB


def enhance_col_with_thumbs(
    db: BaseDB, df: pd.DataFrame, dbr_col: str, thumb_col_name="thumb"
) -> pd.DataFrame:

    thumbs = []
    g = tqdm.tqdm(df[dbr_col], desc=f"Fetching thumbnails for {thumb_col_name}")
    for s in g:
        g.set_description(f"Fetching thumbnails for {thumb_col_name}: '{s}'")
        s = f"<{s.replace('dbr:', 'http://dbpedia.org/resource/')}>"
        res = db.query(f"""PREFIX dbr: <http://dbpedia.org/resource/>
                            PREFIX dbo: <http://dbpedia.org/ontology/>
                            SELECT ?thumb ?thumb_lc WHERE {{
                                {s} dbo:thumbnail_original ?thumb .
                                BIND (LCASE(STR(?thumb)) AS ?thumb_lc) 
                                FILTER (STRENDS(?thumb_lc, ".png") || STRENDS(?thumb_lc, ".jpg") || STRENDS(?thumb_lc, ".jpeg"))
                            }} LIMIT 1""")
        # print(f"Query result for {s}: {res}")
        if len(res) > 0:
            thumbs.append(res["thumb"].values[0])
        else:
            thumbs.append(None)
    df[thumb_col_name] = thumbs
    return df


with db_qlever as db:
    easy_results = enhance_col_with_thumbs(db, easy_results, "s")
    hard_results = enhance_col_with_thumbs(
        db, hard_results, "s", thumb_col_name="thumb_s"
    )
    hard_results = enhance_col_with_thumbs(
        db, hard_results, "r", thumb_col_name="thumb_r"
    )



2026-04-24 14:42:23,169 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-24 14:42:23,170 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz


2026-04-24 14:42:23,170 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-24 14:42:23,171 - INFO - Stopping server!
2026-04-24 14:42:23,248 - ERROR - Command failed with return code 1
2026-04-24 14:42:23,248 - INFO - Starting QLever server on port 25943
2026-04-24 14:42:23,248 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-24 14:42:23,250 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:42:24,251 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:42:25,253 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-24 14:42:26,265 - INFO - Server is up and resp

In [34]:
hard_results.to_csv(Path("scratch") / "dbpedia_hard_results.csv", index=False)
easy_results.to_csv(Path("scratch") / "dbpedia_easy_results.csv", index=False)

In [35]:
easy_results.loc[0, 'thumb'].split('/')[-1]

'Plastiki_hull_closeup.jpg'

## Token

You need to set your Oauth tokens in the local `.env`:
```sh
MEDIA_WIKI_TOKEN="..."
MEDIA_WIKI_SECRET="..."
MEDIA_WIKI_ACCESS_TOKEN="..."
MEDIA_WIKI_ACCESS_SECRET="..."
```
You can register a new token [here](https://meta.wikimedia.org/wiki/Special:OAuthConsumerRegistration/propose/oauth1a).

In [36]:
import requests
from requests_oauthlib import OAuth1
import dotenv
from utils.datasets.dbpedia_utils.helpers import USER_AGENT

dotenv.load_dotenv()  # to load MEDIAWIKI_TOKEN from .env file

auth = OAuth1(
    os.getenv("MEDIA_WIKI_TOKEN"),
    os.getenv("MEDIA_WIKI_SECRET"),
    os.getenv("MEDIA_WIKI_ACCESS_TOKEN"),
    os.getenv("MEDIA_WIKI_ACCESS_SECRET"),
)   

In [37]:
auth.client

<Client client_key=eea9f51d9008d744c272c3ece851b184, client_secret=****, resource_owner_key=914b284deef1f1f0525ea803d55914b0, resource_owner_secret=****, signature_method=HMAC-SHA1, signature_type=AUTH_HEADER, callback_uri=None, rsa_key=None, verifier=None, realm=None, encoding=utf-8, decoding=utf-8, nonce=None, timestamp=None>

In [38]:
url=f"https://commons.wikimedia.org/wiki/Special:FilePath/{easy_results.loc[0, 'thumb'].split('/')[-1]}?width=330"
resp = requests.get(
    url,
    auth=auth,
    headers={"User-Agent": USER_AGENT},
)

In [47]:
import pywikibot

pywikibot.config.usernames['commons']['commons'] = "Dakantz"
# set user-agent

authenticate = (
    os.getenv("MEDIA_WIKI_TOKEN"),
    os.getenv("MEDIA_WIKI_SECRET"),
    os.getenv("MEDIA_WIKI_ACCESS_TOKEN"),
    os.getenv("MEDIA_WIKI_ACCESS_SECRET"),
)
pywikibot.config.authenticate['commons.wikimedia.org'] = authenticate
pywikibot.config.user_agent = USER_AGENT
site = pywikibot.Site('commons', 'commons')
site.login()

In [53]:
site.allimages()

In [68]:
from PIL import Image
fname = easy_results.loc[0, "thumb"].split("/")[-1]
img = pywikibot.FilePage(site, fname)

img.download(filename=fname, url_width=330)
img = Image.open(fname)

In [71]:
from utils.datasets.dbpedia_utils import image_formatter


def thumbs_to_pil(
    thumbs: pd.Series, scratch_dir=Path("../scratch/dbpedia_thumbs")
) -> pd.Series:

    scratch_dir.mkdir(parents=True, exist_ok=True)
    pil_images = []
    g = tqdm.tqdm(thumbs, desc="Fetching thumbnails")
    for thumb in g:
        g.set_description(f"Fetching thumbnail for {thumb}...")
        # url_thumb = get_wc_thumb(thumb)

        fname = thumb.split("/")[-1]
        out_f = scratch_dir / fname
        if not out_f.exists():
            img = pywikibot.FilePage(site, fname)
            img.download(filename=out_f, url_width=330)
        img = Image.open(out_f)
        pil_images.append(img)
    return pd.Series(pil_images, index=thumbs.index)


easy_results["thumb_img"] = thumbs_to_pil(easy_results["thumb"])
hard_results["thumb_s_img"] = thumbs_to_pil(hard_results["thumb_s"])
hard_results["thumb_r_img"] = thumbs_to_pil(hard_results["thumb_r"])

Fetching thumbnail for https://upload.wikimedia.org/wikipedia/commons/0/04/Plastiki_hull_closeup.jpg...:   0%|          | 0/10 [00:00<?, ?it/s]Sleeping for 60.0 seconds, 2026-04-24 15:01:31
Fetching thumbnail for http://upload.wikimedia.org/wikipedia/commons/5/54/2006Boston088.jpg...:  50%|█████     | 5/10 [01:11<00:39,  7.81s/it]         WARNING: Http response status 429
Fetching thumbnail for http://upload.wikimedia.org/wikipedia/commons/5/54/2006Boston088.jpg...:  50%|█████     | 5/10 [01:14<01:14, 14.86s/it]


FileNotFoundError: [Errno 2] No such file or directory: '../scratch/dbpedia_thumbs/2006Boston088.jpg'

In [ ]:
out_dir = Path("../scratch/results/")


def to_tex_with_thumbs(
    df,
    t_cols=["thumb_img"],
    col_mapping={
        "s": "Ship",
        "r": "Railway",
        "thumb_img_tex": "Thumbnail",
        "dist": "Distance",
    },
    base_dir="figures/generated",
    sub_dir="dbpedia/media",
    out_name="dbpedia_easy_results.tex",
    col_format ="lp{3cm}r"
):
    out_dir.mkdir(parents=True, exist_ok=True)
    results_dir = Path(out_dir) / sub_dir
    results_dir.mkdir(parents=True, exist_ok=True)
    fig_dir = Path(base_dir) / sub_dir
    for t_col in t_cols:
        df[f"{t_col}_tex"] = df[t_col]
        
        for i, r in df.iterrows():
            if r[t_col] is not None and isinstance(r[t_col], Image.Image):
                fname = f"{t_col}_{i}.png"
                fig_file = fig_dir / fname
                im_file = results_dir / fname
                df.at[i, f"{t_col}_tex"] = f"\\includegraphics[width=3cm]{{{fig_file}}}"
                print(f"Saving image for row {i} to {im_file} and referencing as {fig_file} in LaTeX", r[t_col])
                img: Image = r[t_col]
                img.save(im_file)
            else:
                df.at[i, f"{t_col}_tex"] = "No image"
    df_renamed = df.rename(columns=col_mapping)
    print(f"Renamed columns for LaTeX: {df_renamed.columns}")
    allowed_cols = [c for c in list(col_mapping.values()) if c in df_renamed.columns]
    print(f"Allowed columns for LaTeX output: {allowed_cols}")
    df_renamed = df_renamed[allowed_cols]
    df_renamed.set_index(allowed_cols[0], inplace=True)
    df_renamed.style.to_latex(
        buf= results_dir / out_name,
        column_format=col_format,
    )


to_tex_with_thumbs(easy_results)

Saving image for row 0 to ../scratch/results/dbpedia/media/thumb_img_0.png and referencing as figures/generated/dbpedia/media/thumb_img_0.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x247 at 0x7FF50012B5C0>
Saving image for row 1 to ../scratch/results/dbpedia/media/thumb_img_1.png and referencing as figures/generated/dbpedia/media/thumb_img_1.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x217 at 0x7FF500128050>
Saving image for row 2 to ../scratch/results/dbpedia/media/thumb_img_2.png and referencing as figures/generated/dbpedia/media/thumb_img_2.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x250 at 0x7FF4FBFE3360>
Renamed columns for LaTeX: Index(['Ship', 'thumb_emb', 'Distance', 'thumb', 'thumb_img', 'Thumbnail'], dtype='str')
Allowed columns for LaTeX output: ['Ship', 'Thumbnail', 'Distance']
